# Imports

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
from datetime import timedelta
import scipy.stats as stats
import rasterio
from rasterio.transform import from_origin
from rasterio.features import rasterize
import geopandas as gpd
from shapely.geometry import Point
import requests
import random
import os
import math
import copy
import sys
import gc
from io import StringIO

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from numpy.lib.stride_tricks import sliding_window_view

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import shap

import seaborn as sns

In [3]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
from datetime import timedelta
import scipy.stats as stats
import rasterio
from rasterio.transform import from_origin
from rasterio.features import rasterize
import geopandas as gpd
from shapely.geometry import Point
import requests
import random
import os
import math
import copy
import sys
import gc
from io import StringIO

In [4]:
from __future__ import annotations

import copy
import random
import time
from dataclasses import asdict, dataclass, field, replace
from typing import Any, Hashable, Mapping, Optional, Sequence

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from numpy.lib.stride_tricks import sliding_window_view

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import shap

import seaborn as sns

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Populate data_dict

In [7]:
path_to_data = "/content/drive/MyDrive/Year 2/Fall 2025/HONOR 3700/Data/Github/"
num_cases = 23

In [8]:
data_dict = {}

In [9]:
# Populate the dfs
for i in range(num_cases):
    if (i != 6):
        continue
    case_dict = {}
    case_dict["data_df"] = pd.read_csv(path_to_data + str(i) + "_data.csv")
    case_dict["metadata_df"] = pd.read_csv(path_to_data + str(i) + "_metadata.csv")

    data_dict[i] = case_dict

In [10]:
data_dict.keys()

dict_keys([6])

In [11]:
# Fine Tuning Stuff
# path_to_fine_tuning = "/content/drive/MyDrive/Year 2/Fall 2025/HONOR 3700/Data/Github/Fine Tuning"
# fine_tune_index = num_cases
# fine_tune_case_dict = {}
# fine_tune_case_dict["data_df"] = pd.read_csv(path_to_data + '6' + "_data.csv")
# fine_tune_case_dict["metadata_df"] = pd.read_csv(path_to_data + '6' + "_metadata.csv")

# data_dict[fine_tune_index] = fine_tune_case_dict

# num_cases += 1

In [12]:
# Populate lookups
for i in range(num_cases):
    if (i != 6):
        continue
    case_dict = data_dict[i]
    unique_points_df = case_dict['data_df'].drop_duplicates(subset=['fid'], keep='first')[["fid", "row_index", "col_index"]]
    case_dict["unique_points"] = unique_points_df
    case_dict["lookup"] = {(r,c):pid for r,c,pid in zip(unique_points_df.row_index, unique_points_df.col_index, unique_points_df.fid)}


In [13]:
# Expect size to be an odd integer
def get_patches(df, size, lookup):

    patches = []

    for r,c in zip(df.row_index, df.col_index):

        valid = True
        patch = []

        for dr in range(-(size // 2) , (size // 2)+1):
            row_vals = []
            for dc in range(-(size // 2) , (size // 2)+1):

                key = (r+dr, c+dc)

                if key not in lookup:
                    valid = False
                    break

                row_vals.append(lookup[key])

            if not valid:
                break

            patch.append(row_vals)

        if valid:
            patches.append(patch)

    patches = np.array(patches)

    return patches

In [14]:
PATCH_SIZE = 5

In [15]:
# Populate patches
for i in range(num_cases):
    if (i != 6):
        continue
    case_dict = data_dict[i]
    index_patches = get_patches(case_dict["unique_points"], PATCH_SIZE, case_dict["lookup"])
    case_dict["index_patches"] = index_patches

In [17]:
for i in range(num_cases):
    if (i != 6):
        continue
    if (len(data_dict[i]['index_patches']) == 0):
        print(i)

In [18]:
# Convert time in metadata to datetime, sort
for i in range(num_cases):
    if (i != 6):
        continue
    case_dict = data_dict[i]
    case_dict['metadata_df']['time'] = pd.to_datetime(case_dict['metadata_df']['time'])
    case_dict['metadata_df'] = case_dict['metadata_df'].sort_values(by='time', ascending=True)

In [19]:
def add_time_features(df):
    if df.empty:
        return
    df = df.copy()

    # Convert time strings to datetime
    t = pd.to_datetime(df["time"])

    # Basic date features
    df["year"] = t.dt.year
    df["month"] = t.dt.month
    df["day_of_year"] = t.dt.dayofyear

    # Seasonality
    df["seasonality"] = np.sin(
        2 * np.pi * df["day_of_year"] / 365
    )

    # Difference from previous observation, in days
    # df["diff_prev"] = (
    #     t.diff()
    #     .dt.total_seconds()
    #     .div(86400)
    #     .fillna(0)
    # )

    # Difference from first observation, in days
    # df["diff_first"] = (
    #     t - t.iloc[0]
    # ).dt.total_seconds().div(86400)

    return df

In [20]:
# Convert time in metadata to datetime, sort
for i in range(num_cases):
    if (i != 6):
        continue
    case_dict = data_dict[i]
    case_dict['metadata_df'] = add_time_features(case_dict['metadata_df'])

In [21]:
# Copy data dict that has non-zero rows for data and index_patches
data_dict_filtered = {
    i: case_dict
    for i, case_dict in data_dict.items()
    if (len(case_dict.get('unique_points', [])) > 0 and len(case_dict['index_patches'] > 0))
}

In [23]:
# Remove empty cases
list_of_empty_patches = [0, 2, 9, 16, 19]
for case_id in list_of_empty_patches:
    if (i != 6):
        continue
    _ = data_dict.pop(case_id)

In [24]:
data_dict.keys()

dict_keys([6])

In [25]:
similarity_url = "https://raw.githubusercontent.com/thomaslu678/Praxis-Lab-25-26/refs/heads/main/assets/studies/Case%20ID%20%2B%20Similarity.csv"
similarity_df = pd.read_csv(similarity_url)

In [26]:
fine_tune_row = pd.DataFrame([{'ID': 23, 'similarity': 1.0}])

# 2. Concatenate them together
similarity_df = pd.concat([similarity_df, fine_tune_row], ignore_index=True)

In [27]:
# Temp populate similiarity scores
for i in data_dict.keys():
    if (i != 6):
        continue
    case_dict = data_dict[i]
    case_dict['similarity_score'] = (similarity_df[similarity_df['ID'] == i]['similarity'].item())

In [28]:
for i in data_dict.keys():
    if (i != 6):
        continue
    data_dict[i]['metadata_df'] = data_dict[i]['metadata_df'].reset_index(drop=True)

In [29]:
bad_data_url = "https://raw.githubusercontent.com/thomaslu678/Praxis-Lab-25-26/refs/heads/main/assets/points/bad_data_fine_tune.csv"
bad_data_df = pd.read_csv(bad_data_url)

In [30]:
def remove_bad_data(data_dict, bad_data_df):
    report = []

    for case_id, bad_case_df in bad_data_df.groupby("case_id"):
        case_key = int(case_id)

        if case_key not in data_dict:
            report.append({
                "case_id": case_key,
                "bad_rows": len(bad_case_df),
                "matched": 0,
                "removed": 0,
                "unmatched": len(bad_case_df),
            })
            continue

        case_dict = data_dict[case_key]
        data_df = case_dict["data_df"]

        # Primary keys to remove
        bad_keys = set(
            zip(bad_case_df["fid"], bad_case_df["mission_id"])
        )

        # Identify rows matching those keys
        row_keys = list(zip(data_df["fid"], data_df["mission_id"]))
        mask = pd.Series(row_keys, index=data_df.index).isin(bad_keys)

        removed = int(mask.sum())

        # Update the dataframe in-place within the nested dict
        case_dict["data_df"] = data_df.loc[~mask].copy()

        report.append({
            "case_id": case_key,
            "bad_rows": len(bad_case_df),
            "matched": removed,
            "removed": removed,
            "unmatched": len(bad_case_df) - removed,
        })

    report_df = pd.DataFrame(report)

    return data_dict, report_df

In [31]:
data_dict, report_df = remove_bad_data(data_dict, bad_data_df)

In [32]:
report_df

,case_id,bad_rows,matched,removed,unmatched
0,1,1537,0,0,1537
1,3,7226,0,0,7226
2,4,2016,0,0,2016
3,5,4886,0,0,4886
4,6,2994,2994,2994,0
5,7,4443,0,0,4443
6,8,833,0,0,833
7,9,23,0,0,23
8,10,1872,0,0,1872
9,11,1996,0,0,1996


In [33]:
def populate_index_patch_metadata(data_dict):
    for case_id in data_dict.keys():
        case_data = data_dict[case_id]

        data_df = case_data["data_df"]
        metadata_df = case_data["metadata_df"].reset_index(drop=True)

        # Normalize FIDs once
        data_fids = data_df["fid"].astype(str)
        data_missions = data_df["mission_id"]

        # Build:
        #   mission_id -> set of FIDs available for that mission
        mission_to_fids = (
            data_df.assign(fid=data_fids)
            .groupby("mission_id")["fid"]
            .agg(set)
            .to_dict()
        )

        # Get the original patches
        index_patches = case_data["index_patches"]

        new_index_patches = []

        for index_patch in index_patches:
            # Flatten 2D patch and normalize FIDs
            patch_fids = set(str(fid) for fid in index_patch.flatten())

            # Find missions that contain every FID in the patch
            valid_mission_ids = [
                mission_id
                for mission_id, mission_fids in mission_to_fids.items()
                if patch_fids.issubset(mission_fids)
            ]

            # Convert mission IDs to metadata_df indices
            metadata_indices = metadata_df.index[
                metadata_df["mission_id"].isin(valid_mission_ids)
            ].tolist()

            new_index_patches.append({
                "data_indices": index_patch,
                "metadata_indices": metadata_indices,
            })

        # Replace original patches in place
        case_data["index_patches"] = new_index_patches

In [34]:
populate_index_patch_metadata(data_dict)

In [35]:
data_dict.keys()

dict_keys([6])

In [48]:
data_dict[6].keys()

dict_keys(['data_df', 'metadata_df', 'unique_points', 'lookup', 'index_patches', 'similarity_score'])

# Instantiate model with weights

# Load datasets for tree patches

## Process zonal stats

In [118]:
zonal_stats_df = pd.read_csv('/content/drive/MyDrive/Year 2/Fall 2025/HONOR 3700/Data/Github/Trees/zonal_stats.csv')

In [119]:
zonal_stats_df

,grid_id,row_index,col_index,red_count,red_sum,red_mean,green_count,green_sum,green_mean,blue_count,blue_sum,blue_mean,nir_count,nir_sum,nir_mean
0,8,0,8,2500,0.000000,0.000000,2500,0.000000,0.000000,2500,0.000000,0.000000,2500,0.000000,0.000000
1,9,0,9,2500,0.000000,0.000000,2500,0.000000,0.000000,2500,0.000000,0.000000,2500,0.000000,0.000000
2,10,0,10,2500,0.000000,0.000000,2500,0.000000,0.000000,2500,0.000000,0.000000,2500,0.000000,0.000000
3,11,0,11,2500,0.000000,0.000000,2500,0.000000,0.000000,2500,0.000000,0.000000,2500,0.000000,0.000000
4,12,0,12,2500,0.000000,0.000000,2500,0.000000,0.000000,2500,0.000000,0.000000,2500,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6154,8469,104,45,2500,441.566650,0.176627,2500,416.443407,0.166577,2500,287.780685,0.115112,2500,606.918517,0.242767
6155,8470,104,46,2500,433.673856,0.173470,2500,410.755080,0.164302,2500,282.782324,0.113113,2500,594.004600,0.237602
6156,8471,104,47,2500,409.950501,0.163980,2500,390.911524,0.156365,2500,265.530274,0.106212,2500,551.322975,0.220529
6157,8472,104,48,2500,406.788552,0.162715,2500,385.810098,0.154324,2500,257.019813,0.102808,2500,567.551559,0.227021


In [120]:
print(zonal_stats_df['red_count'].unique(),
      zonal_stats_df['green_count'].unique(),
      zonal_stats_df['blue_count'].unique(),
      zonal_stats_df['nir_count'].unique())

[2500] [2500] [2500] [2500]


## Process RGBALL

In [121]:
rgbn_df = pd.read_csv('/content/drive/MyDrive/Year 2/Fall 2025/HONOR 3700/Data/Github/Trees/rgbn_grid_radii.csv')

In [122]:
rgbn_df

,Red,Green,Blue,NIR,grid_id,radii_id
0,0.209046,0.194738,0.113210,0.272854,3179.0,1
1,0.204406,0.190381,0.107538,0.272601,3179.0,1
2,0.214384,0.198910,0.123468,0.273802,3179.0,1
3,0.209956,0.192596,0.106904,0.273536,3179.0,1
4,0.201147,0.187533,0.105403,0.268976,3179.0,1
...,...,...,...,...,...,...
355488,0.165053,0.141445,0.037082,0.244995,1568.0,3008
355489,0.169394,0.145628,0.052126,0.245158,1568.0,3008
355490,0.163303,0.147246,0.060376,0.241471,1568.0,3008
355491,0.165974,0.145628,0.060376,0.241471,1568.0,3008


In [123]:
rgbn_df['grid_id'].isna().sum()

np.int64(601)

In [124]:
rgbn_df = rgbn_df.dropna()

In [125]:
rgbn_df.isna().sum()

,0
Red,0
Green,0
Blue,0
NIR,0
grid_id,0
radii_id,0


In [126]:
rgbn_df['grid_id'].dtype

dtype('float64')

In [127]:
rgbn_df['grid_id'] = rgbn_df['grid_id'].astype(int)

/tmp/ipykernel_1746/554811330.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rgbn_df['grid_id'] = rgbn_df['grid_id'].astype(int)


In [128]:
rgbn_df['grid_id']

,grid_id
0,3179
1,3179
2,3179
3,3179
4,3179
...,...
355488,1568
355489,1568
355490,1568
355491,1568


## Process radii df

In [129]:
radii_df = pd.read_csv('/content/drive/MyDrive/Year 2/Fall 2025/HONOR 3700/Data/Github/Trees/trees_species_dbh_grid.csv')

In [130]:
radii_df.rename(columns={'fid': 'radii_id'}, inplace=True)

In [131]:
radii_df

,radii_id,SPP,DBH,grid_id
0,1,Zelkova serrata,10,3179
1,2,Zelkova species,12,3260
2,3,Zelkova species,9,3341
3,4,Zelkova species,10,3341
4,5,Zelkova species,9,3341
...,...,...,...,...
3003,3004,Gleditsia triacanthos,13,949
3004,3005,Tilia cordata,19,525
3005,3006,Pyrus calleryana,2,1567
3006,3007,Pyrus calleryana,2,1568


In [132]:
radii_df.isna().sum()

,0
radii_id,0
SPP,0
DBH,0
grid_id,0


## Process grids

In [133]:
zonal_stats_df.columns

Index(['grid_id', 'row_index', 'col_index', 'red_count', 'red_sum', 'red_mean',
       'green_count', 'green_sum', 'green_mean', 'blue_count', 'blue_sum',
       'blue_mean', 'nir_count', 'nir_sum', 'nir_mean'],
      dtype='object')

In [134]:
grids_df = zonal_stats_df[['grid_id', 'row_index', 'col_index']].copy()

In [135]:
grids_df

,grid_id,row_index,col_index
0,8,0,8
1,9,0,9
2,10,0,10
3,11,0,11
4,12,0,12
...,...,...,...
6154,8469,104,45
6155,8470,104,46
6156,8471,104,47
6157,8472,104,48


In [136]:
grids_df['grid_id'].isna().sum()

np.int64(0)

In [137]:
np.min(grids_df['grid_id'])

8

In [138]:
np.max(grids_df['grid_id'])

8473

## Process radii to grid id

In [139]:
radii_df.columns

Index(['radii_id', 'SPP', 'DBH', 'grid_id'], dtype='object')

In [140]:
radii_to_grid_df = radii_df[['radii_id', 'grid_id']].copy()

In [141]:
radii_to_grid_df

,radii_id,grid_id
0,1,3179
1,2,3260
2,3,3341
3,4,3341
4,5,3341
...,...,...
3003,3004,949
3004,3005,525
3005,3006,1567
3006,3007,1568


In [142]:
radii_to_grid_df['grid_id'].isna().sum()

np.int64(0)

In [143]:
radii_to_grid_df['grid_id'].dtype

dtype('int64')

In [144]:
radii_to_grid_df['grid_id'] = radii_to_grid_df['grid_id'].astype(int)

In [145]:
radii_to_grid_df['grid_id'].dtype

dtype('int64')

## Confirm alignment

In [146]:
# zonal_stats_df
# grids_df
# radii_df
# rgbn_df

In [147]:
print("Grids: ", np.min(zonal_stats_df['grid_id']), np.max(zonal_stats_df['grid_id']))

Grids:  8 8473


In [148]:
print("Grids: ", np.min(grids_df['grid_id']), np.max(grids_df['grid_id']))

Grids:  8 8473


In [149]:
print("Radii: ", np.min(radii_df['radii_id']), np.max(radii_df['radii_id']))

Radii:  1 3008


In [150]:
print("Grids: ", np.min(rgbn_df['grid_id']), np.max(rgbn_df['grid_id']))

Grids:  469 8464


In [151]:
print("Radii: ", np.min(rgbn_df['radii_id']), np.max(rgbn_df['radii_id']))

Radii:  1 3008


In [152]:
# 1. Check if all in zonal_stats_df
check1 = rgbn_df['grid_id'].isin(zonal_stats_df['grid_id']).all()

# 2. Check if all in grids_df
check2 = rgbn_df['grid_id'].isin(grids_df['grid_id']).all()

# 3. Final check: True if both conditions are met
all_present = check1 and check2

print(f"All present: {all_present}")

All present: True


In [153]:
print(f"All present: {rgbn_df['radii_id'].isin(radii_df['radii_id']).all()}")

All present: True


In [154]:
print(len(rgbn_df[rgbn_df['radii_id'].isin(radii_df['radii_id'])]))
print(len(rgbn_df[~rgbn_df['radii_id'].isin(radii_df['radii_id'])]))
print(len(rgbn_df))

354892
0
354892


In [155]:
rgbn_df = rgbn_df[rgbn_df['radii_id'].isin(radii_df['radii_id'])]

In [156]:
print(f"All present: {rgbn_df['radii_id'].isin(radii_df['radii_id']).all()}")

All present: True


In [157]:
print(f"All present: {radii_df['radii_id'].isin(radii_to_grid_df['radii_id']).all()}")

All present: True


# Prepare processing tree patches

In [158]:
# zonal_stats_df
# grids_df
# radii_df
# rgbn_df

In [159]:
filtered_radii_df = radii_df[radii_df['radii_id'].isin(rgbn_df['radii_id'])].copy()

In [160]:
radii_df

,radii_id,SPP,DBH,grid_id
0,1,Zelkova serrata,10,3179
1,2,Zelkova species,12,3260
2,3,Zelkova species,9,3341
3,4,Zelkova species,10,3341
4,5,Zelkova species,9,3341
...,...,...,...,...
3003,3004,Gleditsia triacanthos,13,949
3004,3005,Tilia cordata,19,525
3005,3006,Pyrus calleryana,2,1567
3006,3007,Pyrus calleryana,2,1568


In [161]:
filtered_radii_df

,radii_id,SPP,DBH,grid_id
0,1,Zelkova serrata,10,3179
1,2,Zelkova species,12,3260
2,3,Zelkova species,9,3341
3,4,Zelkova species,10,3341
4,5,Zelkova species,9,3341
...,...,...,...,...
3003,3004,Gleditsia triacanthos,13,949
3004,3005,Tilia cordata,19,525
3005,3006,Pyrus calleryana,2,1567
3006,3007,Pyrus calleryana,2,1568


In [162]:
filtered_radii_df.isna().sum()

,0
radii_id,0
SPP,0
DBH,0
grid_id,0


In [163]:
filtered_radii_df['grid_id'].dtype

dtype('int64')

In [164]:
filtered_radii_df['grid_id'] = filtered_radii_df['grid_id'].astype(int)

In [165]:
filtered_radii_df['grid_id'].dtype

dtype('int64')

In [166]:
filtered_radii_df

,radii_id,SPP,DBH,grid_id
0,1,Zelkova serrata,10,3179
1,2,Zelkova species,12,3260
2,3,Zelkova species,9,3341
3,4,Zelkova species,10,3341
4,5,Zelkova species,9,3341
...,...,...,...,...
3003,3004,Gleditsia triacanthos,13,949
3004,3005,Tilia cordata,19,525
3005,3006,Pyrus calleryana,2,1567
3006,3007,Pyrus calleryana,2,1568


# Process raw data into grid/radii id before/after

In [167]:
# Initialize list to collect results
rows = []

In [168]:
subset = rgbn_df[rgbn_df['radii_id'] == 1]

In [169]:
subset

,Red,Green,Blue,NIR,grid_id,radii_id
0,0.209046,0.194738,0.113210,0.272854,3179,1
1,0.204406,0.190381,0.107538,0.272601,3179,1
2,0.214384,0.198910,0.123468,0.273802,3179,1
3,0.209956,0.192596,0.106904,0.273536,3179,1
4,0.201147,0.187533,0.105403,0.268976,3179,1
...,...,...,...,...,...,...
112,0.223045,0.209867,0.144034,0.276180,3179,1
113,0.203237,0.186965,0.101950,0.275078,3179,1
114,0.210689,0.198542,0.120355,0.275691,3179,1
115,0.216111,0.201571,0.129756,0.273536,3179,1


In [170]:
pix = subset[subset['grid_id'] == 4713]

In [171]:
pix

,Red,Green,Blue,NIR,grid_id,radii_id


In [172]:
zonal_stats_df[zonal_stats_df['grid_id'] == 4713].iloc[0]

,3321
grid_id,4713.000000
row_index,58.000000
col_index,15.000000
red_count,2500.000000
red_sum,482.317792
red_mean,0.192927
green_count,2500.000000
green_sum,444.910008
green_mean,0.177964
blue_count,2500.000000


In [173]:
pix['Red'].sum()

np.float64(0.0)

In [174]:
# Loop over each radius (fid)
for fid in filtered_radii_df['radii_id'].unique():

    # Subset rgbn_df for this radii_id
    subset = rgbn_df[rgbn_df['radii_id'] == fid]

    # Get unique grid_ids for this radius
    grid_ids = subset['grid_id'].unique()

    for gid in grid_ids:

        # Subset pixels for this grid_id + radii_id
        pix = subset[subset['grid_id'] == gid]

        # Get zonal stats row
        zs_row = zonal_stats_df[zonal_stats_df['grid_id'] == gid].iloc[0]

        # BEFORE values (means already computed)
        before_red = zs_row['red_mean']
        before_green = zs_row['green_mean']
        before_blue = zs_row['blue_mean']
        before_nir = zs_row['nir_mean']

        # ORIGINAL sums
        red_sum = zs_row['red_sum']
        green_sum = zs_row['green_sum']
        blue_sum = zs_row['blue_sum']
        nir_sum = zs_row['nir_sum']

        red_count = zs_row['red_count']
        green_count = zs_row['green_count']
        blue_count = zs_row['blue_count']
        nir_count = zs_row['nir_count']

        # Remove pixels inside radius
        removed_count = len(pix)

        # Subtract pixel values
        new_red_sum = red_sum - pix['Red'].sum()
        new_green_sum = green_sum - pix['Green'].sum()
        new_blue_sum = blue_sum - pix['Blue'].sum()
        new_nir_sum = nir_sum - pix['NIR'].sum()

        # AFTER means
        after_red = new_red_sum / (red_count - removed_count) if red_count > 0 else None
        after_green = new_green_sum / (green_count - removed_count) if green_count > 0 else None
        after_blue = new_blue_sum / (blue_count - removed_count) if blue_count > 0 else None
        after_nir = new_nir_sum / (nir_count - removed_count) if nir_count > 0 else None

        # Append result
        rows.append({
            'grid_id': gid,
            'radii_id': fid,
            'before_red': before_red,
            'after_red': after_red,
            'before_green': before_green,
            'after_green': after_green,
            'before_blue': before_blue,
            'after_blue': after_blue,
            'before_nir': before_nir,
            'after_nir': after_nir
        })

In [175]:
# Create final dataframe
result_df = pd.DataFrame(rows)

In [176]:
result_df

,grid_id,radii_id,before_red,after_red,before_green,after_green,before_blue,after_blue,before_nir,after_nir
0,3179,1,0.206899,0.206892,0.193059,0.193027,0.115139,0.115065,0.270782,0.270768
1,3260,2,0.195428,0.195552,0.180035,0.180312,0.098392,0.099081,0.262536,0.262468
2,3341,3,0.195303,0.195197,0.179745,0.179697,0.097258,0.097526,0.261336,0.261111
3,3341,4,0.195303,0.195444,0.179745,0.180053,0.097258,0.097853,0.261336,0.261181
4,3341,5,0.195303,0.195530,0.179745,0.180183,0.097258,0.097996,0.261336,0.261305
...,...,...,...,...,...,...,...,...,...,...
4114,950,3004,0.203356,0.203380,0.182451,0.182508,0.137598,0.137682,0.277749,0.277664
4115,525,3005,0.186331,0.186107,0.170221,0.170260,0.080206,0.080281,0.261865,0.261632
4116,1567,3006,0.201844,0.201953,0.190481,0.190616,0.123607,0.123810,0.259652,0.259705
4117,1568,3007,0.191359,0.191453,0.175122,0.175230,0.101810,0.101964,0.258648,0.258696


In [177]:
len(result_df['radii_id'].unique())

2894